In [0]:
silver_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/silver"

customers_silver = spark.read.format("delta").load(
    f"{silver_base}/customers"
)

products_silver = spark.read.format("delta").load(
    f"{silver_base}/products"
)

orders_silver = spark.read.format("delta").load(
    f"{silver_base}/orders"
)

print("Customers:", customers_silver.count())
print("Products:", products_silver.count())
print("Orders:", orders_silver.count())

Customers: 1001
Products: 104
Orders: 9339


In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    sum,
    countDistinct,
    avg
)

daily_sales = (
    orders_silver
    .withColumn("order_date", to_date(col("order_date")))
    .groupBy("order_date")
    .agg(
        sum("total_amount").alias("total_sales"),
        sum("quantity").alias("units_sold"),
        countDistinct("order_id").alias("total_orders"),
        avg("total_amount").alias("average_order_value")
    )
    .orderBy("order_date")
)

In [0]:
display(daily_sales)

order_date,total_sales,units_sold,total_orders,average_order_value
2022-01-01,1119.36,26,10,111.93599999999999
2022-01-02,292.87,13,5,58.574
2022-01-03,1311.8700000000001,33,11,119.2609090909091
2022-01-04,783.6599999999999,26,10,78.36599999999999
2022-01-05,851.02,25,10,85.102
2022-01-06,1043.91,24,13,80.30076923076923
2022-01-07,1252.4900000000002,34,14,89.46357142857144
2022-01-08,821.04,24,10,82.104
2022-01-09,668.3299999999999,21,9,74.25888888888888
2022-01-10,1246.44,27,12,103.87


In [0]:
gold_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/gold"

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/daily_sales")

In [0]:
daily_sales_gold = spark.read.format("delta").load(
    f"{gold_base}/daily_sales"
)

print("Daily Sales rows:", daily_sales_gold.count())

display(daily_sales_gold.limit(20))

Daily Sales rows: 951


order_date,total_sales,units_sold,total_orders,average_order_value
2022-01-01,1119.3600000000001,26,10,111.936
2022-01-02,292.87,13,5,58.574
2022-01-03,1311.8700000000001,33,11,119.2609090909091
2022-01-04,783.6600000000001,26,10,78.36600000000001
2022-01-05,851.0199999999999,25,10,85.10199999999999
2022-01-06,1043.91,24,13,80.30076923076923
2022-01-07,1252.49,34,14,89.46357142857143
2022-01-08,821.04,24,10,82.104
2022-01-09,668.33,21,9,74.25888888888889
2022-01-10,1246.44,27,12,103.87


In [0]:
product_performance = (
    orders_silver.alias("o")
    .join(
        products_silver.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner"
    )
    .groupBy(
        col("p.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.subcategory"),
        col("p.gender"),
        col("p.brand")
    )
    .agg(
        sum(col("o.quantity")).alias("units_sold"),
        sum(col("o.total_amount")).alias("total_sales"),
        countDistinct(col("o.order_id")).alias("total_orders"),
        avg(col("o.unit_price")).alias("average_unit_price")
    )
    .orderBy(col("total_sales").desc())
)

In [0]:
product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/product_performance")

In [0]:
category_sales = (
    orders_silver.alias("o")
    .join(
        products_silver.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner"
    )
    .groupBy(
        col("p.category")
    )
    .agg(
        sum(col("o.quantity")).alias("units_sold"),
        sum(col("o.total_amount")).alias("total_sales"),
        countDistinct(col("o.order_id")).alias("total_orders")
    )
    .orderBy(col("total_sales").desc())
)

In [0]:
category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/category_sales")

In [0]:
customer_sales = (
    orders_silver.alias("o")
    .join(
        customers_silver.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "inner"
    )
    .groupBy(
        col("c.customer_id"),
        col("c.first_name"),
        col("c.last_name"),
        col("c.email"),
        col("c.gender"),
        col("c.city"),
        col("c.state"),
        col("c.country")
    )
    .agg(
        countDistinct(col("o.order_id")).alias("total_orders"),
        sum(col("o.quantity")).alias("total_units"),
        sum(col("o.total_amount")).alias("total_spend"),
        avg(col("o.total_amount")).alias("average_order_value")
    )
    .orderBy(col("total_spend").desc())
)

In [0]:
customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/customer_sales")

In [0]:
print("=" * 50)
print("SHOPSPHERE GOLD TRANSFORMATION")
print("=" * 50)

print(f"Daily Sales rows:          {daily_sales.count()}")
print(f"Product Performance rows: {product_performance.count()}")
print(f"Category Sales rows:       {category_sales.count()}")
print(f"Customer Sales rows:       {customer_sales.count()}")

print("Gold transformations completed successfully.")

SHOPSPHERE GOLD TRANSFORMATION
Daily Sales rows:          951
Product Performance rows: 104
Category Sales rows:       4
Customer Sales rows:       1001
Gold transformations completed successfully.
